# 03 · Bileşen ablasyonu ve duyarlılık (önerilen modelin seçilen hiperparametreleri üzerine tek değişiklik)
Ablasyon: akımsız (noI) · R sıcaklıktan bağımsız · yalnız ohmik kol · tek RC kolu · RC + serbest CNN detector · ohmik yumuşatma yok.
Duyarlılık: W = 128 · ohmik yumuşatma 5 s · τ başlangıcı 1/10/60 s. Seed'ler: `ABL_SEEDS`. Tam model ve Hibrit, 02'deki final koşulardan alınır.

In [ ]:
import sys, os, json, time, warnings
import os; ROOT = os.path.abspath('..')  # repository root (run notebooks from the notebooks/ folder)
SMOKE = False                 # True: çok küçük hızlı deneme (kurulum testi)  False
sys.path.insert(0, ROOT)
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch
from data_utils import *
from models.experiment import *
from models import engine
exp = Experiment(ROOT, smoke=SMOKE)
exp.load_data()
print('Cihaz:', exp.dev, '| W =', exp.cfg['W'])

In [ ]:
sel = exp.load_selected()
abl, sens = ablation_specs(sel)
for stage, specs in [('ablation', abl), ('sensitivity', sens)]:
    for name, spec in specs.items():
        for seed in exp.cfg['ABL_SEEDS']:
            exp.run(name, spec, seed, stage, ablations=RC_ABLATIONS_TEST if spec['family'] == 'rc' else ())
for stage in ['ablation', 'sensitivity']:
    d = exp.collect(stage)
    t = (d.assign(k=np.where(d.temp == '-20degC', '−20°C', 'eğitim sıc.')).groupby(['model', 'k', 'seed']).rmse.mean() * 100).groupby(['model', 'k']).agg(['mean', 'std']).unstack('k')
    print(f'\n{stage}:'); print(t.round(3).to_string())